In [1]:
pip install requests

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing requests library to connect with frankfurter API.
import requests
import datetime
import logging

In [3]:
# Define a new function that helps to bring currencies data as requested.
def get_exchange_rate(base_currency):
    url = f"https://api.frankfurter.dev/v2/rates?base={base_currency}"
    response = requests.get(url)
    if response.status_code != 200:
        raise ValueError(f"API request failed: {response.text}")
    data = response.json()
    return data

currency_rate = get_exchange_rate("EGP")

In [4]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Fetching exchange rates from frankfurter API...")

2026-07-21 16:25:42,057 - INFO - Fetching exchange rates from frankfurter API...


In [5]:
# Check if all currencies rate is up  to date.
today = str(datetime.date.today())
important_currencies = ['USD', 'SAR', 'EGP']
all_updated = True
for rate in currency_rate:
    if rate['quote'] in important_currencies and rate['date'] != today:
        all_updated = False
        logging.warning(f"{rate['quote']} rate is outdated, last updated: {rate['date']}")
if all_updated:
    logging.info('Needed currencies are up to date')

2026-07-21 16:25:42,080 - INFO - Needed currencies are up to date


In [6]:
for rate in currency_rate:
    if rate['quote'] in ['USD', 'SAR'] and rate['rate'] is None:
        raise ValueError(f"{rate['quote']} rate is missing - cannot proceed with comparison")

In [7]:
# Base is the main currency, EGP rate to others.
for rate in currency_rate[:5]:
    print(f"1 {rate['base']} = {rate['rate']} {rate['quote']}, -----> Updated in {rate['date']}")

1 EGP = 0.07192 AED, -----> Updated in 2026-07-21
1 EGP = 1.2908 AFN, -----> Updated in 2026-07-21
1 EGP = 1.6043 ALL, -----> Updated in 2026-07-21
1 EGP = 7.1656 AMD, -----> Updated in 2026-07-21
1 EGP = 0.03506 ANG, -----> Updated in 2026-07-21


In [8]:
# other rates to EGP.
for rate in currency_rate[:5]:
        print(f"1 {rate['quote']} = {round(1 / rate['rate'],2)} {rate['base']}, -> Updated in {rate['date']}")

1 AED = 13.9 EGP, -> Updated in 2026-07-21
1 AFN = 0.77 EGP, -> Updated in 2026-07-21
1 ALL = 0.62 EGP, -> Updated in 2026-07-21
1 AMD = 0.14 EGP, -> Updated in 2026-07-21
1 ANG = 28.52 EGP, -> Updated in 2026-07-21


In [9]:
# How much USD & SAR in EGP.
for rate in currency_rate:
    if rate['quote'] == 'SAR':
        sar_to_egp = 1 / rate['rate']
    elif rate['quote'] == 'USD':
        usd_to_egp = 1 / rate['rate']
print(sar_to_egp)
print(usd_to_egp)

logging.info(f"sar_to_egp: {sar_to_egp}")
logging.info(f"usd_to_egp: {usd_to_egp}")

2026-07-21 16:25:42,170 - INFO - sar_to_egp: 13.616557734204791
2026-07-21 16:25:42,170 - INFO - usd_to_egp: 51.07252298263534


13.616557734204791
51.07252298263534


In [10]:
sar_to_usd = sar_to_egp / usd_to_egp
sar_to_usd

0.2666122004357298

In [11]:
result_direct = 1000 * sar_to_egp
result_direct

13616.55773420479

In [12]:
result_indirect = (1000 * sar_to_usd) * usd_to_egp
result_indirect

13616.557734204789

In [13]:
if result_direct > result_indirect:
    logging.info(f"Direct value as {round(result_direct,4)} is bigger than indirect value, It's {round((result_direct-result_indirect)/result_indirect*100,4)}% bigger than {round(result_indirect,4)}.")
elif result_direct < result_indirect:
    logging.info(f"Indirect value as {round(result_indirect,4)} is bigger than direct value, It's {round((result_indirect-result_direct)/result_direct*100,4)}% bigger than {round(result_direct,4)}.")
else:
    logging.info(f"{round(result_indirect,4)} & {round(result_direct,4)} ARE EQUALS!!!")

2026-07-21 16:25:42,258 - INFO - Direct value as 13616.5577 is bigger than indirect value, It's 0.0% bigger than 13616.5577.
